# JEPA ladder: the Titan-register model with JEPA, a style head and contrastive style

```
"Titan: <x>"  → ModernBERT-base + LoRA → bridge → x prefix (one vector per token) ─────────────────────┐   [GPU 0, frozen]
                                                   │                                                    │
                                                   └→ predictor (6-layer transformer) → ẑ = 8 content   │   [GPU 0, trained]
                                                                                        + 2 style slots ┤
                                                                                                        ▼
                      [<bos>] + x prefix + ẑ + target  →  Gemma 3 4B IT (4-bit) + LoRA  →  y, the Titan register   [GPU 1]

JEPA target:      "Plain: <y>" → the same frozen encoder + bridge → y's token vectors → mean-pooled into 8 slots s(y)
Style target:     the 7 attribute scores of y (titan_style_v1, Part A)
Contrast set:     the same facts as a formal and a near-miss version (titan_style_v1, Part B; 400 train rows)
```

**Four arms on one ladder.** Each arm adds one loss to the arm before it; nothing else differs.

| Arm | Loss | Question |
|---|---|---|
| **control** | CE on y | baseline: the same model and slots, no latent objective |
| **jepa** | + λ_j · (1 − cos(ẑ_content, s(y))) | does predicting the target's latent help? |
| **style** | + λ_s · MSE(style head(ẑ_style), 7 attribute scores of y) | does explicit style supervision help? (arm D of the design) |
| **contrast** | + λ_c · InfoNCE(ẑ_content: s(Titan) against s(formal), s(near-miss)) | does contrastive style separation help? (arm E) |

**Same in all four arms:**
- the stage-1 encoder LoRA and bridge (R16's, frozen);
- Gemma's LoRA reset to zero at the start;
- the predictor's and style head's initial weights, and the 10 slots (8 content + 2 style);
- data, data order, seed, steps, learning rates and early stopping (validation cross-entropy).

**The JEPA target s(y).** The stage-1 autoencoder maps a sentence into Gemma's input space with chrF 98.5. y, encoded as `Plain: <y>` by the same frozen encoder and bridge, is mean-pooled into `K_SLOTS` chunks. The target is fixed, so it cannot collapse, and the cosine is taken after per-dimension standardisation.

**Stage 1 is R16's**, downloaded from the private repo `skamalj/titan-stage1` (`r16/`) and checked: plain Gemma must reconstruct held-out sentences through the bridge. If neither the local files nor the Hub copy is reachable, stage 1 trains here first.

**Scores, per arm, on validation and test:**
- chrF;
- facts kept: numbers, dates, names, negation;
- style:
  - the register's hard rules;
  - **attribute error**: a ridge probe on the frozen latents predicts each output's 7 attributes, compared with the reference's scores;
  - **Titan rate**: a 4-way probe (plain / formal / near-miss / Titan) classifies each output;
- copies of the source;
- the slot test: swap or add noise to ẑ.

**The bar, set before the run:** an arm earns its loss if it beats the arm below it by at least **+2 chrF on test** without losing faithfulness (numbers, dates, names, negation, at most 2 points lower). Style metrics are reported next to it. Only a clear win is worth seeds.

**Checkpoints.** This is a long run (about 2 h per arm). Everything is saved to `/kaggle/working/` **and mirrored to the private Hub repo** (`runs/<RUN_NAME>/`), so even a Kaggle session that ends loses nothing:
- stage 1, if it trains here: a checkpoint every 500 steps;
- each arm's training: one checkpoint per epoch, and the best epoch's weights;
- each arm's scoring: one save per finished part (each split, the slot test, the demo, the samples);
- each finished arm: its results, Gemma LoRA and predictor.

After any stop: restart the kernel and Run All. Finished parts are skipped, and the unfinished one resumes.

In [ ]:
# Cell 0 — Hugging Face token (Gemma 3 is gated): Kaggle secret, or HF_TOKEN in the environment (Modal); never printed

import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    for label in ("HF_TOKEN", "hf_token"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(label)
            break
        except Exception:
            pass
assert HF_TOKEN, "add the Hugging Face token under Add-ons → Secrets (label HF_TOKEN or hf_token) and tick it"


## Cell 1 — Setup, settings, pinned revisions

Two GPUs (Kaggle: *GPU T4 × 2*). The settings that define the experiment are all here.

In [ ]:
# Cell 1 — Packages, settings, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q -U bitsandbytes
!pip install -q "transformers==5.17.0" "peft==0.21.2"   # fills in any missing dependency (e.g. httpx on Modal)

import gc, json, os, random, re, shutil, statistics, time, urllib.error, urllib.request
from collections import Counter

import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, hf_hub_download, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
TOP_REPO, PHINC_REPO = "rvv-karma/English-Hinglish-TOP", "veezbo/phinc"
BRANCH = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/project4-hindi-decoder/data/"
DATA = BRANCH + "titan_v1/"                  # Titan register: 1,360 delivered pairs + h301–h400 (train 1,260 / 100 / 100)
STYLE_DATA = BRANCH + "titan_style_v1/"      # Part A attribute scores (all rows), Part B contrast versions (600 rows)
VAL_SPLIT, VERDICT_SPLIT = "validation", "test"   # early stopping; the A/B verdict (test is held out for it)
MAX_NEW_TOKENS = 128                         # targets run to 60 words
MAX_TARGET_TOKENS = 128
DEMO_INPUTS = [                              # translated in each arm; kept out of every training set
    "I worked all weekend to fix the bug, and nobody noticed.",
    "We can still win this game if we don't give up in the second half.",
    "My brother borrowed my car and returned it with an empty tank.",
    "I'm not angry that you lied; I'm disappointed that you thought you had to.",
    "The project is three weeks late, but the client still wants it by Friday.",
    "I trained for a year, and I missed the medal by one second.",
    # the user's manual test (kept out of all training)
    "I'll be late to the office because of traffic.",
    "I'm running 10 minutes late. Please start without me.",
    "I'm really hungry. Let's get something to eat.",
    "I'm exhausted after today's work. I need some rest.",
    "I'm stuck in a meeting. I'll call you when I'm free.",
]

# The experiment: four arms, each adding one loss to the arm before it; nothing else differs
ARMS = {"control":  {"jepa": 0.0, "style": 0.0, "contrast": 0.0},
        "jepa":     {"jepa": 1.0, "style": 0.0, "contrast": 0.0},
        "style":    {"jepa": 1.0, "style": 0.3, "contrast": 0.0},
        "contrast": {"jepa": 1.0, "style": 0.3, "contrast": 0.5}}
RUN_ARMS = [a for a in os.environ.get("RUN_ARMS", ",".join(ARMS)).split(",") if a]   # Modal: one arm per job
WARM_MAX_EPOCHS, WARM_PATIENCE, WARM_BATCH = 40, 4, 32   # predictor warm-up on the arm's latent losses, Gemma untouched
K_SLOTS, K_STYLE = 8, 2                      # predicted content slots (JEPA target) + style slots (style head)
PRED_LAYERS, PRED_WIDTH, PRED_HEADS = 6, 512, 8   # the standard JEPA predictor: a 6-layer transformer
PRED_LR = 3e-4                               # jepa/style/contrast arms (control ran with 1e-4, see Cell 8)
CONTRAST_TAU = 0.1                           # InfoNCE temperature
BAR_CHRF = 2.0                               # an arm must beat the arm below it by this much on VERDICT_SPLIT
FAITH_SLACK = 0.02                           # ... and lose no more than this on any faithfulness rate

# Checkpoint mirror (private Hub repo): a Kaggle session that ends loses /kaggle/working, not these
HUB_REPO = "skamalj/titan-stage1"
STAGE1_HUB = "r16"                           # R16's stage-1 encoder LoRA + bridge in that repo
RUN_NAME = "jepa-ladder-v1"                  # change it to start a fresh run instead of resuming this one

SEED = 42
DTYPE = torch.float32
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()   # Modal: the job's volume folder

N_GPU = torch.cuda.device_count()
assert N_GPU >= 1, "this notebook needs a GPU (Kaggle: GPU T4 x2; Modal: one L40S)"
ENC_DEVICE, DEC_DEVICE = ((torch.device("cuda:0"), torch.device("cuda:1")) if N_GPU >= 2
                          else (torch.device("cuda:0"), torch.device("cuda:0")))
login(HF_TOKEN)
api = HfApi(token=HF_TOKEN)
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
REV.update({r: api.dataset_info(r).sha for r in (TOP_REPO, PHINC_REPO)})
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), (
    f"pinned versions not active (transformers {transformers.__version__}, peft {peft.__version__}): restart the kernel "
    "and run from the top")


def hub_path(local):
    return f"runs/{RUN_NAME}/{os.path.relpath(local, WORK)}"


def hub_put(local, remote=None):
    """Mirror a file or folder from /kaggle/working to the private repo. Never fatal: the local copy stays."""
    remote = remote or hub_path(local)
    try:
        if os.path.isdir(local):
            api.upload_folder(repo_id=HUB_REPO, folder_path=local, path_in_repo=remote,
                              commit_message=f"{RUN_NAME}: {remote}")
        else:
            api.upload_file(repo_id=HUB_REPO, path_or_fileobj=local, path_in_repo=remote,
                            commit_message=f"{RUN_NAME}: {remote}")
        return True
    except Exception as e:
        print(f"  (Hub mirror failed for {remote}: {type(e).__name__}; the local copy is kept)")
        return False


def hub_get(local, remote=None):
    """Restore a missing file or folder from the mirror into /kaggle/working. True if it exists locally afterwards."""
    if os.path.exists(local):
        return True
    remote = remote or hub_path(local)
    try:
        files = [f for f in api.list_repo_files(HUB_REPO) if f == remote or f.startswith(remote + "/")]
        for f in files:
            dst = local if f == remote else os.path.join(local, os.path.relpath(f, remote))
            os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
            shutil.copy(hf_hub_download(HUB_REPO, f, token=HF_TOKEN), dst)
        if files:
            print(f"  restored {remote} from the Hub mirror")
        return bool(files)
    except Exception as e:
        print(f"  (Hub restore failed for {remote}: {type(e).__name__})")
        return False


def hub_delete(local):
    """Remove a finished arm's training checkpoints from the mirror (they are large); never fatal."""
    try:
        api.delete_file(hub_path(local), repo_id=HUB_REPO, commit_message=f"{RUN_NAME}: drop {hub_path(local)}")
    except Exception:
        pass


try:
    HUB_OK = bool(api.model_info(HUB_REPO))
except Exception:
    HUB_OK = False
print("checkpoint mirror:", HUB_REPO if HUB_OK else "NOT reachable (local checkpoints only)", "| run:", RUN_NAME)
print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "revisions": REV,
       "arms": ARMS, "slots": (K_SLOTS, K_STYLE), "run_arms": RUN_ARMS, "work": WORK, "predictor": (PRED_LAYERS, PRED_WIDTH, PRED_HEADS)})

## Cell 2 — Titan data and style signals

`DATA` (Cell 1) is `data/titan_v1/`: an everyday English sentence → the same facts in the **Titan register**. The register is calm, immense and patient: short declaratives, no contractions, one to five sentences, at most 60 words, one of ten rhetorical moves, and every time, name, number and request kept. Encoder input `"Titan: <English>"`.

| Split | Rows | Use |
|---|---|---|
| `train` | 1,260 | 1,160 delivered + 100 handwritten (h301–h400) |
| `validation` | 100 | early stopping (cross-entropy), scores |
| `test` | 100 | the verdict |

`STYLE_DATA` is `data/titan_style_v1/`:
- **attributes:** 7 scores, 0–1, for every source and every target (formality, grandiosity, dramatic tone, philosophical tone, authoritative tone, elevated vocabulary, aggression). Scorer agreement on a blind rescore: mean difference 0.02–0.06.
- **contrast:** a `formal` and a `near_miss` version with the same facts for 600 sources (all of validation and test, and 400 train rows, 40 per move).

Each record gets `style` (its target's 7 scores), `source_style`, and, where present, `contrast` (formal, near_miss).

In [ ]:
# Cell 2 — Titan data and style signals

def fetch_jsonl(name, base=None):
    return [json.loads(l) for l in urllib.request.urlopen((base or DATA) + name).read().decode("utf-8").splitlines() if l]


try:
    MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
    NAME = {p: v["name"] for p, v in MANIFEST["personas"].items()}
except urllib.error.HTTPError:
    NAME = {"titan": "Titan"}


def persona_record(r):
    pid = r.get("persona_id", "titan")
    return {**r, "persona": pid, "references": r.get("references") or [r["target"]],
            "enc_input": f"{NAME[pid]}: {r['source']}"}


SPLITS = {}
for s in ("train", "validation", "test"):
    try:
        SPLITS[s] = [persona_record(r) for r in fetch_jsonl(f"{s}.jsonl")]
    except urllib.error.HTTPError:
        assert s not in ("train", VAL_SPLIT, VERDICT_SPLIT), f"{DATA}{s}.jsonl is required"
TRAIN, VAL = SPLITS["train"], SPLITS[VAL_SPLIT]
PERSONA = TRAIN[0]["persona"]
EVAL_SPLITS = [s for s in ("validation", "test") if s in SPLITS]

ATTRS = ["formality", "grandiosity", "dramatic_tone", "philosophical_tone", "authoritative_tone",
         "elevated_vocabulary", "aggression"]
STYLE = {r["id"]: r for r in fetch_jsonl("attributes.jsonl", STYLE_DATA)}
CONTRAST = {r["id"]: r for r in fetch_jsonl("contrast.jsonl", STYLE_DATA)}
for rows in SPLITS.values():
    for r in rows:
        r["style"] = [STYLE[r["id"]]["target_style"][k] for k in ATTRS]
        r["source_style"] = [STYLE[r["id"]]["source_style"][k] for k in ATTRS]
        if r["id"] in CONTRAST:
            r["contrast"] = (CONTRAST[r["id"]]["formal"], CONTRAST[r["id"]]["near_miss"])
print({s: len(rs) for s, rs in SPLITS.items()}, "| data:", DATA)
print("with contrast versions:", {s: sum("contrast" in r for r in rs) for s, rs in SPLITS.items()})
print("target attribute means:", {k: round(statistics.mean(r["style"][i] for r in TRAIN), 2) for i, k in enumerate(ATTRS)})
print("first validation pair:", VAL[0]["source"], "->", VAL[0]["target"])

## Cell 3 — Public corpora for stage 1

Used for two things:
- the **stage-1 check**: 40 held-out sentences reconstructed through the bridge;
- **stage 1 itself**, if no earlier stage-1 files are found.

As in notebook 11:
- English side only (reconstruction): Hinglish-TOP (Apache-2.0) and PHINC (CC BY 4.0), cleaned;
- sentences resembling a Titan evaluation or demo sentence are dropped;
- 200 sentences held out from each corpus.

In [ ]:
# Cell 3 — Corpora

def download(repo, path, name):
    url = f"https://huggingface.co/datasets/{repo}/resolve/{REV[repo]}/{path}"
    local = f"/tmp/{name}"
    if not os.path.exists(local):
        urllib.request.urlretrieve(url, local)
    return local


top = pd.read_parquet(download(TOP_REPO, "data/train-00000-of-00001.parquet", "top_train.parquet"))
phinc = pd.read_csv(download(PHINC_REPO, "hinglish_english_parallel_corpus.csv", "phinc.csv"))
DEVA = re.compile(r"[ऀ-ॿ]")


def clean_tweet(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", str(t))
    t = re.sub(r"[@#]\w+", " ", t)
    return re.sub(r"\s+", " ", t).strip(" .…:-")


def usable(en, hi):
    ne, nh = len(en.split()), len(hi.split())
    return 3 <= ne <= 30 and 3 <= nh <= 30 and not DEVA.search(hi) and 0.5 <= nh / ne <= 2.0


pairs = {"top_human": [], "top_synthetic": [], "phinc": []}
for en, hi, gen in zip(top["en"], top["hi_en"], top["generated_by"]):
    en, hi = str(en).strip(), str(hi).strip()
    if usable(en, hi):
        pairs["top_human" if gen == "human" else "top_synthetic"].append((en, hi))
for hi, en in zip(phinc["Sentence"], phinc["English_Translation"]):
    en, hi = clean_tweet(en), clean_tweet(hi)
    if usable(en, hi):
        pairs["phinc"].append((en, hi))

words = lambda t: set(re.findall(r"[a-z0-9']+", t.lower()))
EVAL_WORDS = [words(r["source"]) for s in SPLITS if s != "train" for r in SPLITS[s]] + [words(t) for t in DEMO_INPUTS]
pairs = {k: [(en, en) for en, _ in v] for k, v in pairs.items()}      # reconstruction: English in, same English out


def leaks(en):
    w = words(en)
    return any(len(w & e) / len(w | e) >= 0.6 for e in EVAL_WORDS if w | e)


rng = random.Random(SEED)
for k in pairs:
    rng.shuffle(pairs[k])
seen, dropped = set(), Counter()


def take(items, n):
    """The first n usable pairs of a shuffled list: not seen before, not resembling a Titan evaluation or demo sentence."""
    out = []
    for en, hi in items:
        if len(out) >= n:
            break
        key = en.lower()
        if key in seen:
            dropped["duplicate"] += 1
        elif leaks(en):
            dropped["resembles an evaluation or demo sentence"] += 1
        else:
            seen.add(key)
            out.append((en, hi))
    return out


CORPUS_TOTAL = 12000                        # stage-1 sentences (used only if stage 1 has to run here)
PHINC_MAX = CORPUS_TOTAL // 2
HELD = [("top", *p) for p in take(pairs["top_human"], 200)] + [("phinc", *p) for p in take(pairs["phinc"], 200)]
train_pairs = [("top", *p) for p in take(pairs["top_human"][200:], len(pairs["top_human"]))]
train_pairs += [("phinc", *p) for p in take(pairs["phinc"][200:], PHINC_MAX)]
train_pairs += [("top", *p) for p in take(pairs["top_synthetic"], max(0, CORPUS_TOTAL - len(train_pairs)))]
rng.shuffle(train_pairs)
train_pairs = train_pairs[:CORPUS_TOTAL]


def corpus_record(src, en, hi):
    return {"corpus": src, "source": en, "target": hi, "enc_input": f"Plain: {en}", "references": [hi]}


CORPUS_TRAIN = [corpus_record(*p) for p in train_pairs]
CORPUS_HELD = [corpus_record(*p) for p in HELD]
del top, phinc, pairs
gc.collect()
print(f"stage 1 pairs {len(CORPUS_TRAIN)} {dict(Counter(r['corpus'] for r in CORPUS_TRAIN))} | held out "
      f"{len(CORPUS_HELD)} | dropped while sampling {dict(dropped)}")

## Cell 4 — Models

In [ ]:
# Cell 4 — ModernBERT on GPU 0; Gemma 3 4B IT (4-bit) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError("GPU memory already in use: restart the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE)
dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
START = dec_tok.bos_token_id
EOS = dec_tok.convert_tokens_to_ids("<end_of_turn>")
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
PAD = dec_tok.pad_token_id
D_ENC, D_DEC = encoder.config.hidden_size, decoder.config.text_config.hidden_size
print(f"encoder width {D_ENC} | decoder width {D_DEC} | start {START}, end {EOS}")

## Cell 5 — Bridge, LoRA, predictor, the forward pass, translation, scores

- **Bridge and LoRAs:** as in notebook 11.
- **Predictor:** a 6-layer pre-norm transformer over `[x prefix ; 10 learned queries]`. It projects 2560 → `PRED_WIDTH`, adds learned positions, and its outputs at the query positions are projected back to 2560: the slots ẑ, 8 content and 2 style.
- **Style head:** a linear map from the mean of the 2 style slots to the 7 attributes (trained only in the style and contrast arms; present in all).
- **Gemma's input:** `[<bos>] + x prefix + ẑ + target + <end_of_turn>`; the loss is on the target only.
- **Translation:** `generate()` gets `<bos>` and one placeholder per vector, and the embedding hook writes the prefix and ẑ in.

**Faithfulness checks** (automatic, per row; each rate is over the rows where the check applies). The fact checks are the dataset's own validator (`raw/validate_titan.py`):
- **numbers:** every number and clock time in the source appears in the output, as digits or words ("9:30" may be "half past nine");
- **dates:** every weekday and month;
- **names:** every capitalised word that doesn't start a sentence;
- **negation:** a source negation is kept;
- **style** (reported, not part of the verdict): no contractions, no "!", at most 60 words and 5 sentences.

Questions and added negations aren't checked, because the register turns questions into statements and adds "not" freely. Cell 5 prints every rate on the reference targets too: that is the ceiling.

In [ ]:
# Cell 5 — Bridge, LoRA, predictor, forward pass, translation, scores

import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


class Bridge(nn.Module):
    """Encoder states -> vectors in Gemma's input-embedding space: a 2-layer MLP, then RMSNorm at the embeddings' scale."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.mlp(h))


class Predictor(nn.Module):
    """x prefix (n vectors) -> K slots: a 6-layer transformer over [x ; K learned queries], read at the queries."""

    def __init__(self, d, k, width, layers, heads, max_len=128):
        super().__init__()
        self.k = k
        self.inp = nn.Linear(d, width)
        self.pos = nn.Parameter(torch.randn(max_len + k, width) * 0.02)
        self.queries = nn.Parameter(torch.randn(k, width) * 0.02)
        layer = nn.TransformerEncoderLayer(width, heads, 4 * width, dropout=0.1, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.body = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(width)
        self.out = nn.Linear(width, d)

    def forward(self, xs):
        T = max(len(x) for x in xs)
        x = torch.stack([F.pad(v, (0, 0, 0, T - len(v))) for v in xs])
        pad = torch.tensor([[False] * len(v) + [True] * (T - len(v)) for v in xs], device=x.device)
        h = self.inp(x) + self.pos[:T]
        q = (self.queries + self.pos[T:T + self.k]).expand(len(xs), -1, -1)
        mask = torch.cat([pad, torch.zeros(len(xs), self.k, dtype=torch.bool, device=x.device)], 1)
        out = self.body(torch.cat([h, q], 1), src_key_padding_mask=mask)
        return self.out(self.norm(out[:, T:]))


torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
predictor = Predictor(D_DEC, K_SLOTS + K_STYLE, PRED_WIDTH, PRED_LAYERS, PRED_HEADS).to(ENC_DEVICE)
style_head = nn.Linear(D_DEC, len(ATTRS)).to(ENC_DEVICE)        # mean of the style slots -> 7 attributes
decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["Wqkv", "Wo", "Wi"]))
EMBED = dec_model.get_input_embeddings()

with torch.no_grad():                             # start the prefix at the scale of real token embeddings
    sample = dec_tok(" ".join(r["target"] for r in TRAIN[:50]), add_special_tokens=False)["input_ids"]
    EMB_RMS = EMBED(torch.tensor(sample, device=DEC_DEVICE)).float().pow(2).mean(-1).sqrt().mean().item()
    bridge.norm.weight.fill_(EMB_RMS)

GROUPS = {"encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
          "bridge": list(bridge.parameters()),
          "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n],
          "predictor": list(predictor.parameters()) + list(style_head.parameters())}
for params in GROUPS.values():
    for p in params:
        p.requires_grad_(True)
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "decoder LoRA": 5e-5, "predictor": PRED_LR}
STAGE_GROUPS = {1: ["encoder LoRA", "bridge", "decoder LoRA"],       # reconstruction (only if stage 1 runs here)
                2: ["decoder LoRA", "predictor"]}                    # all arms: encoder and bridge frozen
STAGE = {"now": 1}


def set_trainable(stage):
    STAGE["now"] = stage
    for g, params in GROUPS.items():
        for p in params:
            p.requires_grad_(g in STAGE_GROUPS[stage])


def train_mode(stage):
    encoder.train(stage == 1)
    bridge.train(stage == 1)
    predictor.train(stage == 2)
    dec_model.train()


def eval_mode():
    encoder.eval()
    bridge.eval()
    predictor.eval()
    dec_model.eval()


def reset_decoder_lora():
    """Gemma's LoRA back to B = 0 (Gemma as released): stage 2 learns the voice from scratch (R16's reset path)."""
    for n, p in dec_model.named_parameters():
        if "lora_B" in n:
            p.data.zero_()


def prefixes(texts):
    """Encoder input texts -> one prefix per text (bridge vectors for its real tokens), on GPU 0."""
    with torch.set_grad_enabled(torch.is_grad_enabled() and STAGE["now"] == 1):
        t = enc_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
        p = bridge(h)
    return [p[i, :int(n)] for i, n in enumerate(t["attention_mask"].sum(1))]


def heads(records, slots=True, slot_records=None, noise=0.0):
    """Per record: x prefix, followed by the predicted slots ẑ (from slot_records if given: the swap test)."""
    x = prefixes([r["enc_input"] for r in records])
    if not slots:
        return x, None
    z = predictor(x if slot_records is None else prefixes([r["enc_input"] for r in slot_records]))
    if Z_SPACE["std"] is not None:          # the predictor speaks the standardised latent; Gemma reads it at real scale
        z = z * Z_SPACE["std"] + Z_SPACE["mean"]
    if noise:
        z = z + noise * z.detach().pow(2).mean(-1, keepdim=True).sqrt() * torch.randn_like(z)
    return [torch.cat([a, b]) for a, b in zip(x, z)], z


def embed_ids(ids):
    return EMBED(torch.tensor(ids, device=DEC_DEVICE))


def loss_on(records, slots=True):
    """Gemma's cross-entropy on the targets; also returns ẑ (B, K, D on GPU 0) for the JEPA loss."""
    hs, z = heads(records, slots)
    rows, labels = [], []
    for r, h in zip(records, hs):
        head = torch.cat([embed_ids([START]), h.to(DEC_DEVICE)])
        tgt = dec_tok(r["target"], add_special_tokens=False)["input_ids"][:MAX_TARGET_TOKENS] + [EOS]
        rows.append(torch.cat([head, embed_ids(tgt)]))
        labels.append([-100] * len(head) + tgt)
    T = max(len(x) for x in rows)
    emb = torch.stack([F.pad(x, (0, 0, 0, T - len(x))) for x in rows])
    mask = torch.tensor([[1] * len(x) + [0] * (T - len(x)) for x in rows], device=DEC_DEVICE)
    lab = torch.tensor([y + [-100] * (T - len(y)) for y in labels], device=DEC_DEVICE)
    ce = dec_model(inputs_embeds=emb, attention_mask=mask, labels=lab, token_type_ids=torch.zeros_like(mask)).loss
    return ce, z


class PrefixSlot:
    vectors = None


SLOT = PrefixSlot()
PLACEHOLDER = dec_tok(" .", add_special_tokens=False)["input_ids"][-1]


def write_prefix(module, args, output):
    """generate() cannot start from embeddings: replace the placeholders' embeddings with the prefix (first pass only)."""
    if SLOT.vectors is not None and output.dim() == 3 and output.shape[1] == 1 + len(SLOT.vectors):
        output = output.clone()
        output[:, 1:] = SLOT.vectors.to(output.dtype)
        SLOT.vectors = None
    return output


EMBED.register_forward_hook(write_prefix)


@torch.no_grad()
def generate(r, slots=True, slot_record=None, noise=0.0, n=1, temperature=None, top_p=0.95):
    eval_mode()
    hs, _ = heads([r], slots, None if slot_record is None else [slot_record], noise)
    p = hs[0].to(DEC_DEVICE)
    ids = torch.tensor([[START] + [PLACEHOLDER] * len(p)], device=DEC_DEVICE)
    outs = []
    for _ in range(n):
        SLOT.vectors = p
        try:
            kw = (dict(do_sample=True, temperature=temperature, top_p=top_p) if temperature else
                  dict(do_sample=False, num_beams=1, temperature=None, top_p=None, top_k=None))
            out = dec_model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=MAX_NEW_TOKENS,
                                     eos_token_id=STOPS, pad_token_id=PAD, **kw)
        finally:
            written = SLOT.vectors is None
            SLOT.vectors = None
        assert written, "the prefix was not written into the prompt"
        new = out[0, ids.shape[1]:].tolist()
        outs.append({"output": dec_tok.decode(new, skip_special_tokens=True).strip(),
                     "stopped": bool(new) and new[-1] in STOPS})
    return outs[0] if n == 1 else outs


def translate(r, **kw):
    return generate(r, slots=STAGE["now"] == 2, **kw)


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


# --- faithfulness checks: the fact checks of the dataset's own validator (raw/validate_titan.py), per row ---
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
ORDINAL = {1: "first", 2: "second", 3: "third", 5: "fifth", 8: "eighth", 9: "ninth", 12: "twelfth"}
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def cardinal(n):
    if n < 20:
        return ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return TENS[t] + ("" if o == 0 else "-" + ONES[o])
    if n < 1000:
        h, r = divmod(n, 100)
        return ONES[h] + " hundred" + ("" if r == 0 else " " + cardinal(r))
    if n < 1000000:
        th, r = divmod(n, 1000)
        return cardinal(th) + " thousand" + ("" if r == 0 else " " + cardinal(r))
    return str(n)


def ordinal(n):
    if n in ORDINAL:
        return ORDINAL[n]
    if n < 20:
        return cardinal(n) + "th"
    if n >= 100:
        head, tail = n - n % 100, n % 100
        return cardinal(head) + ("th" if tail == 0 else " " + ordinal(tail))
    if n % 10 == 0:
        return cardinal(n)[:-1] + "ieth"
    t, o = divmod(n, 10)
    return TENS[t] + "-" + ordinal(o)


def num_words(n):
    c, o = cardinal(n), ordinal(n)
    forms = {c, c.replace("-", " "), c.replace(" hundred ", " hundred and "), o, o.replace("-", " ")}
    if 1000 <= n < 10000 and n % 100 == 0:
        forms.add(cardinal(n // 100) + " hundred")
    forms |= {15: {"quarter past", "a quarter"}, 30: {"half past", "half an hour", "half"},
              45: {"quarter to"}}.get(n, set())
    return forms


has_any = lambda low, forms: any(f in low for f in forms)


def time_ok(h, mm, low):
    H, M = int(h), int(mm)
    nxt = H % 12 + 1
    return (f"{H}:{mm}" in low or (M == 0 and has_any(low, num_words(H)))
            or (has_any(low, num_words(H)) and has_any(low, num_words(M)))
            or (M == 15 and "quarter past" in low and has_any(low, num_words(H)))
            or (M == 30 and "half past" in low and has_any(low, num_words(H)))
            or (M == 45 and "quarter to" in low and has_any(low, num_words(nxt)))
            or (M > 30 and f"{cardinal(60 - M)} to" in low and has_any(low, num_words(nxt))))


def fact_checks(src, out):
    """Numbers (incl. clock times), weekdays/months and mid-sentence names of the source, found in the output."""
    low = out.lower().replace("-", " ")
    numbers, dates, names = [], [], []
    rest = src
    for m in re.finditer(r"\b(\d{1,2}):(\d{2})\b", src):
        numbers.append(time_ok(m.group(1), m.group(2), low))
        rest = rest.replace(m.group(0), " ")
    for m in re.finditer(r"\$?\d[\d,]*(?:\.\d+)?", rest):
        tok = m.group(0).replace("$", "").replace(",", "")
        if "." in tok:
            whole, frac = tok.split(".")
            numbers.append(tok in low or (tok == "0.25" and "quarter" in low) or (tok == "0.5" and "half" in low)
                           or (has_any(low, num_words(int(whole))) and has_any(low, num_words(int(frac)))))
        else:
            numbers.append(tok in low or has_any(low, num_words(int(tok))))
    for d in WEEKDAYS + MONTHS:
        present = (re.search(r"(\d\w*\s+May\b|\bMay\s+\d|\b(in|by|until|of|before|after|this|next|last)\s+May\b)", src)
                   if d == "may" else re.search(rf"\b{d}\b", src, re.I))
        if present:
            dates.append(d in low)
    for m in re.finditer(r"(?<![.?!]\s)(?<!^)\b([A-Z][a-z]{2,})\b", src):
        if m.group(1).lower() not in WEEKDAYS + MONTHS:
            names.append(m.group(1).lower() in low)
    return {"numbers_ok": all(numbers) if numbers else None, "dates_ok": all(dates) if dates else None,
            "names_ok": all(names) if names else None}


NEG = re.compile(r"\b(not|no|never|nobody|nothing|none|neither|nor|without|cannot|nowhere)\b|n't", re.I)
CONTRACTION = re.compile(r"\b(it|he|she|that|what|there|here|who|let|where|how|this)'s\b|\b\w+'(re|ve|ll|m)\b|n't\b"
                         r"|\b(i|he|she|we|they|you|it|that|who)'d\b", re.I)


def faithfulness(src, out):
    """Facts kept (the dataset's checks), negation kept, and the register's hard style rules."""
    n_sent = len([x for x in re.split(r"(?<=[.?!])\s+", out.strip()) if x])
    return {**fact_checks(src, out),
            "negation_ok": bool(NEG.search(out)) if NEG.search(src) else None,
            "style_ok": not CONTRACTION.search(out) and "!" not in out and len(out.split()) <= 60 and n_sent <= 5}


FAITH = ["numbers_ok", "dates_ok", "names_ok", "negation_ok"]          # facts: the verdict's faithfulness
STYLE = ["style_ok"]                                                   # the register's hard rules: reported only
norm = lambda t: re.sub(r"\W+", " ", t.lower()).strip()
train_targets = {r["target"].strip() for r in TRAIN}


def rate(rows, key):
    vals = [x[key] for x in rows if x.get(key) is not None]
    return round(sum(vals) / len(vals), 3) if vals else None


def summarise(rows):
    s = {"n": len(rows), "chrf": round(statistics.mean(x["chrf"] for x in rows), 1),
         "stopped": round(sum(x["stopped"] for x in rows) / len(rows), 3),
         "distinct_outputs": len({x["output"] for x in rows}),
         "copies_of_source": sum(norm(x["output"]) == norm(x["source"]) for x in rows),
         "copies_of_training_targets": sum(x["output"].strip() in train_targets for x in rows)}
    s.update({k: rate(rows, k) for k in FAITH + STYLE})
    if "attr_err" in rows[0]:
        s["attr_err"] = round(statistics.mean(x["attr_err"] for x in rows), 3)
        s["titan_rate"] = round(sum(x["style_class"] == "titan" for x in rows) / len(rows), 3)
    return s


def evaluate(records, **kw):
    rows = []
    for r in records:
        out = translate(r, **kw)
        rows.append({**{k: r.get(k) for k in ("id", "persona", "corpus", "source", "enc_input", "references")}, **out,
                     "chrf": max(chrf(out["output"], ref) for ref in r["references"]),
                     **faithfulness(r["source"], out["output"])})
        if "style" in r:
            rows[-1]["ref_style"] = r["style"]
    if PROBES and all("ref_style" in x for x in rows):
        add_style(rows)
    return rows, summarise(rows)


Z_SPACE = {"mean": None, "std": None}       # set in Cell 7 from the training targets (not used by the control arm)
PROBES = {}                                  # the style probes (Cell 7); scoring adds style metrics once they exist


def show(rows, n, seed=7):
    for x in random.Random(seed).sample(rows, min(n, len(rows))):
        print(f"  in : {x['enc_input']}\n  out: {x['output']}" + ("" if x["stopped"] else "  [did not stop]") +
              f"\n  ref: {x['references'][0]}\n")


def make_optimizer(steps, stage):
    opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": LRS[g]} for g in STAGE_GROUPS[stage]], weight_decay=0.0)
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / max(1, steps)))


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


assert faithfulness("Tell Sana I paid 18,000 by 9:30 on Friday, not later.",
                    "Sana will know. Eighteen thousand, paid by half past nine on Friday. Not later.") == \
    {"numbers_ok": True, "dates_ok": True, "names_ok": True, "negation_ok": True, "style_ok": True}
REFERENCE = {sp: {k: rate([faithfulness(r["source"], r["target"]) for r in SPLITS[sp]], k) for k in FAITH + STYLE}
             for sp in EVAL_SPLITS}
print("the checks on the reference targets (the ceiling for the model):", REFERENCE)
print({k: f"{sum(p.numel() for p in v) / 1e6:.2f} M" for k, v in GROUPS.items()}, f"| prefix scale {EMB_RMS:.3f}")

## Cell 6 — Stage 1: R16's autoencoder, or train it here

Stage 2 freezes the encoder and bridge and resets Gemma's LoRA. So stage 1 contributes only `encoder-lora/` and `bridge.pt`. They are looked for in this order:
1. `/kaggle/working/`;
2. the private Hub repo `skamalj/titan-stage1`, folder `r16/` (R16's stage 1, chrF 98.5);
3. otherwise stage 1 trains here, as in notebook 11: 12,000 sentences, batches of 8, about 2 h. A checkpoint is saved every 500 steps and mirrored to the Hub, and a restarted kernel resumes from it.

**Check, in every case:** with Gemma's LoRA at zero, 40 held-out sentences are reconstructed through the bridge. Below chrF 50 the notebook stops.

The first run of this notebook trained stage 1 here, because R16's files were gone. It reached chrF 17.0 at step 500 and 29.2 at step 1,000, against R16's 80.7 and 98.3: the copy never clicked, as in R14. So R16's stage 1 is now kept on the Hub.

In [ ]:
# Cell 6 — Stage 1 (load or train), then the reconstruction check

from safetensors.torch import load_file
from peft import set_peft_model_state_dict

ENC_FILE, BRIDGE_FILE = f"{WORK}/encoder-lora/adapter_model.safetensors", f"{WORK}/bridge.pt"
if not (os.path.exists(ENC_FILE) and os.path.exists(BRIDGE_FILE)):           # R16's stage 1 from the private repo
    hub_get(f"{WORK}/encoder-lora", f"{STAGE1_HUB}/encoder-lora")
    hub_get(BRIDGE_FILE, f"{STAGE1_HUB}/bridge.pt")
S1_QUICK = [r for r in CORPUS_HELD if r["corpus"] == "top"][:20] + [r for r in CORPUS_HELD if r["corpus"] == "phinc"][:20]
set_trainable(1)
if os.path.exists(ENC_FILE) and os.path.exists(BRIDGE_FILE):
    res = set_peft_model_state_dict(encoder, load_file(ENC_FILE, device=str(ENC_DEVICE)))
    assert not [k for k in res.unexpected_keys if "lora_" in k], res.unexpected_keys
    assert not [k for k in res.missing_keys if "lora_" in k], "encoder-lora/ does not match this encoder's LoRA"
    bridge.load_state_dict(torch.load(BRIDGE_FILE, map_location=ENC_DEVICE, weights_only=True)["bridge"])
    S1_SOURCE = {"loaded": [ENC_FILE, BRIDGE_FILE],
                 "saved_at": [time.ctime(os.path.getmtime(f)) for f in (ENC_FILE, BRIDGE_FILE)]}
    S1_LOG = None
else:
    S1_BATCH, S1_EVAL_EVERY = 8, 500
    S1_CKPT = f"{WORK}/stage1_jepa_control_ckpt.pt"
    s1_steps = -(-len(CORPUS_TRAIN) // S1_BATCH)
    opt, sched = make_optimizer(s1_steps, 1)
    start = 0
    hub_get(S1_CKPT)
    ck = torch.load(S1_CKPT, weights_only=False) if os.path.exists(S1_CKPT) else None
    if ck is not None:
        restore(ck["params"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        S1_LOG, start = ck["log"], ck["step"]
        del ck
        gc.collect()
        print(f"resuming stage 1 from the checkpoint at step {start}")
    else:
        S1_LOG = []
    t0, run = time.perf_counter(), []
    for step in range(start + 1, s1_steps + 1):
        train_mode(1)
        loss, _ = loss_on(CORPUS_TRAIN[(step - 1) * S1_BATCH: step * S1_BATCH], slots=False)
        loss.backward()
        opt.step()
        sched.step()
        opt.zero_grad()
        run.append(loss.item())
        if step % S1_EVAL_EVERY == 0 or step == s1_steps:
            _, quick = evaluate(S1_QUICK)
            S1_LOG.append({"step": step, "train_loss": round(sum(run) / len(run), 4), "chrf_40": quick["chrf"]})
            run = []
            torch.save({"step": step, "params": snapshot(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "log": S1_LOG}, S1_CKPT)
            hub_put(S1_CKPT)
            print(S1_LOG[-1], f"| {time.perf_counter() - t0:.0f}s | checkpoint saved")
    del opt, sched
    encoder.save_pretrained(f"{WORK}/encoder-lora")
    torch.save({"bridge": bridge.state_dict(), "prefix_scale": EMB_RMS}, BRIDGE_FILE)
    hub_put(f"{WORK}/encoder-lora")
    hub_put(BRIDGE_FILE)
    S1_SOURCE = {"trained_here": True, "steps": s1_steps}
    gc.collect()
    torch.cuda.empty_cache()

reset_decoder_lora()                                   # the check uses Gemma as released, as stage 2 starts
S1_ROWS, S1_CHECK = evaluate(S1_QUICK)
print("stage 1:", S1_SOURCE, "\nreconstruction with Gemma's LoRA at zero, 40 held-out:", S1_CHECK)
show(S1_ROWS, 4)
assert S1_CHECK["chrf"] >= 50, "the stage-1 files do not carry sentences: delete encoder-lora/ and bridge.pt and rerun"

## Cell 7 — Latent targets, style losses, style probes

**Latents.** For every sentence that needs one (Titan targets, sources, contrast versions, and later each output):
1. `Plain: <sentence>` goes through the frozen encoder and bridge.
2. Only the sentence's own tokens are kept, without `[CLS] Plain :` and `[SEP]`.
3. They are mean-pooled into `K_SLOTS` chunks.

The per-dimension mean and standard deviation come from the training targets.

**Losses on the predicted slots ẑ** (B × 10 × 2560):
- **JEPA** (jepa, style, contrast): 1 − cos per slot between the standardised content slots `ẑ[:, :8]` and s(y), averaged.
- **Style** (style, contrast): MSE between `style_head(mean(ẑ[:, 8:]))` and y's 7 attribute scores.
- **Contrast** (contrast, on rows that have versions): InfoNCE at temperature 0.1. The content slots must sit closer to s(Titan y) than to s(formal) and s(near-miss) of the same facts, by mean standardised slot cosine.

**Style probes, for scoring the outputs of every arm.** Both are fitted on the training rows' frozen latents (mean of the standardised slots). The regularisation is chosen on validation.
- **attribute probe:** ridge regression to the 7 scores, fitted on 2,520 train sentences (sources and targets). An output's **attribute error** is the mean absolute difference between its predicted scores and the reference target's scores.
- **style class probe:** ridge one-vs-rest over plain / formal / near-miss / Titan, fitted on the 400 train contrast rows. An output's **Titan rate** is how often it is classified Titan.

The probes' validation accuracy is printed, so their own reliability is visible.

In [ ]:
# Cell 7 — Latent targets, style losses, style probes

HEAD_LEN = len(enc_tok("Plain:")["input_ids"]) - 1          # [CLS] Plain : (without [SEP])
assert enc_tok("Plain: the day")["input_ids"][:HEAD_LEN] == enc_tok("Plain:")["input_ids"][:HEAD_LEN]
set_trainable(2)
eval_mode()
TARGETS = {}


@torch.no_grad()
def cache_targets(texts, batch=32):
    """Pooled stage-1 latent (K_SLOTS × D, on CPU) of each sentence, cached by text."""
    todo = sorted({t for t in texts if t not in TARGETS})
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        for t, p in zip(chunk, prefixes([f"Plain: {t}" for t in chunk])):
            y = p[HEAD_LEN:-1]
            if len(y) == 0:
                y = p
            TARGETS[t] = F.adaptive_avg_pool1d(y.T[None].float(), K_SLOTS)[0].T.cpu()


ALL_RECORDS = [r for rows in SPLITS.values() for r in rows]
cache_targets([r["target"] for r in ALL_RECORDS] + [r["source"] for r in ALL_RECORDS]
              + [t for r in ALL_RECORDS for t in r.get("contrast", ())])
_all = torch.cat([TARGETS[r["target"]] for r in TRAIN])
T_MEAN = _all.mean(0).to(ENC_DEVICE)
T_STD = _all.std(0).clamp_min(1e-3 * _all.std(0).mean().item()).to(ENC_DEVICE)
del _all
std = lambda v: (v.to(ENC_DEVICE) - T_MEAN) / T_STD
Z_SPACE.update(mean=T_MEAN, std=T_STD)      # from here on, std(ẑ) is exactly the predictor's own output


def jepa_cos(z, records):
    """Per-slot cosine between standardised content slots and s(y): (B, K_SLOTS)."""
    t = torch.stack([TARGETS[r["target"]] for r in records])
    return F.cosine_similarity(std(z[:, :K_SLOTS]), std(t), dim=-1)


def style_pred(z):
    return style_head(z[:, K_SLOTS:].mean(1))


def style_loss(z, records):
    return F.mse_loss(style_pred(z), torch.tensor([r["style"] for r in records], device=ENC_DEVICE))


def contrast_sims(z, records):
    """Rows with contrast versions: mean standardised slot cosine of the content slots to s(Titan), s(formal),
    s(near-miss) of the same facts: (n, 3), or None if the batch has no such row."""
    idx = [i for i, r in enumerate(records) if "contrast" in r]
    if not idx:
        return None
    cands = torch.stack([torch.stack([TARGETS[records[i]["target"]], TARGETS[records[i]["contrast"][0]],
                                      TARGETS[records[i]["contrast"][1]]]) for i in idx])       # n, 3, K, D
    zc = std(z[idx, :K_SLOTS])[:, None]                                                        # n, 1, K, D
    return F.cosine_similarity(zc, std(cands), dim=-1).mean(-1)                                # n, 3


def contrast_loss(z, records):
    sims = contrast_sims(z, records)
    if sims is None:
        return None
    return F.cross_entropy(sims / CONTRAST_TAU, torch.zeros(len(sims), dtype=torch.long, device=sims.device))


@torch.no_grad()
def val_losses(records, batch=8):
    eval_mode()
    ce = cos = mse = 0.0
    wins, n_c = 0, 0
    for i in range(0, len(records), batch):
        chunk = records[i:i + batch]
        loss, z = loss_on(chunk)
        ce += loss.item() * len(chunk)
        cos += jepa_cos(z, chunk).mean().item() * len(chunk)
        mse += style_loss(z, chunk).item() * len(chunk)
        sims = contrast_sims(z, chunk)
        if sims is not None:
            wins += int((sims.argmax(1) == 0).sum())
            n_c += len(sims)
    n = len(records)
    return {"validation_ce": round(ce / n, 4), "validation_jepa_cos": round(cos / n, 4),
            "validation_style_mse": round(mse / n, 4), "validation_contrast_acc": round(wins / n_c, 3) if n_c else None}


# --- style probes on frozen latents (for scoring outputs) ---
feat = lambda texts: torch.stack([std(TARGETS[t]).mean(0) for t in texts]).double()     # N × D


def ridge_fit(X, Y, alpha):
    mx, my = X.mean(0), Y.mean(0)
    Xc = X - mx
    W = torch.linalg.solve(Xc.T @ Xc + alpha * torch.eye(X.shape[1], device=X.device, dtype=X.dtype), Xc.T @ (Y - my))
    return lambda Z: (Z - mx) @ W + my


def pick(fit_X, fit_Y, val_X, val_Y, score):
    best = None
    for alpha in (1.0, 10.0, 100.0, 1000.0, 10000.0):
        f = ridge_fit(fit_X, fit_Y, alpha)
        sc = score(f(val_X), val_Y)
        if best is None or sc > best[0]:
            best = (sc, alpha, f)
    return best


tr_txt = [r["source"] for r in TRAIN] + [r["target"] for r in TRAIN]
tr_att = torch.tensor([r["source_style"] for r in TRAIN] + [r["style"] for r in TRAIN], dtype=torch.float64,
                      device=ENC_DEVICE)
va_txt = [r["source"] for r in VAL] + [r["target"] for r in VAL]
va_att = torch.tensor([r["source_style"] for r in VAL] + [r["style"] for r in VAL], dtype=torch.float64,
                      device=ENC_DEVICE)
neg_mae = lambda P, Y: -(P - Y).abs().mean().item()
a_mae, a_alpha, ATTR_PROBE = pick(feat(tr_txt), tr_att, feat(va_txt), va_att, neg_mae)
va_tgt_mae = (ATTR_PROBE(feat([r["target"] for r in VAL])) - va_att[len(VAL):]).abs().mean(0)

CLASSES = ["plain", "formal", "near_miss", "titan"]


def class_set(rows):
    rows = [r for r in rows if "contrast" in r]
    txt = [t for r in rows for t in (r["source"], r["contrast"][0], r["contrast"][1], r["target"])]
    lab = [c for _ in rows for c in range(4)]
    return txt, torch.nn.functional.one_hot(torch.tensor(lab, device=ENC_DEVICE), 4).double()


c_tr_txt, c_tr_y = class_set(TRAIN)
c_va_txt, c_va_y = class_set(VAL)
acc = lambda P, Y: (P.argmax(1) == Y.argmax(1)).double().mean().item()
c_acc, c_alpha, CLASS_PROBE = pick(feat(c_tr_txt), c_tr_y, feat(c_va_txt), c_va_y, acc)
PROBES.update({"attr": ATTR_PROBE, "class": CLASS_PROBE})


@torch.no_grad()
def add_style(rows):
    """Adds each output's predicted attributes, attribute error against the reference, and style class."""
    eval_mode()
    outs = [x["output"] or "." for x in rows]
    cache_targets(outs)
    X = feat(outs)
    A, C = PROBES["attr"](X), PROBES["class"](X).argmax(1).tolist()
    for x, a, c in zip(rows, A.tolist(), C):
        x["style_pred"] = [round(v, 3) for v in a]
        x["attr_err"] = round(statistics.mean(abs(p - q) for p, q in zip(a, x["ref_style"])), 4)
        x["style_class"] = CLASSES[c]


ref_rows = [{"output": r["target"], "ref_style": r["style"]} for r in VAL]
add_style(ref_rows)
REF_STYLE = {"attr_err": round(statistics.mean(x["attr_err"] for x in ref_rows), 3),
             "titan_rate": round(sum(x["style_class"] == "titan" for x in ref_rows) / len(ref_rows), 3)}
_s = [TARGETS[r["target"]] for r in TRAIN[:200]]
pair_cos = statistics.mean(F.cosine_similarity(std(_s[i]), std(_s[i + 1]), dim=-1).mean().item() for i in range(0, 198, 2))
print(f"{len(TARGETS)} latents cached | standardised cosine between unrelated training targets {pair_cos:.3f}")
print(f"attribute probe: alpha {a_alpha:g}, validation MAE {-a_mae:.3f} (sources + targets); on targets per attribute",
      {k: round(v, 3) for k, v in zip(ATTRS, va_tgt_mae.tolist())})
print(f"style class probe: alpha {c_alpha:g}, validation accuracy {c_acc:.3f} over {CLASSES} (chance 0.25)")
print("reference targets as outputs (the ceiling):", REF_STYLE)
del _s

## Cell 8 — One arm

`run_arm(name)` runs one arm with the loss weights `ARMS[name]`:
1. **Start.** Restore the stage-1 weights and the predictor's and style head's initial weights, then reset Gemma's LoRA. This is identical for every arm.
2. **Train** Gemma's LoRA, the predictor and the style head: micro-batches of 4, 2-step accumulation, up to 6 epochs, the same data order per epoch. Loss = CE + λ_j · JEPA + λ_s · style + λ_c · contrast. A term with weight 0 is logged but not trained on.
3. **Early stopping** after 2 epochs without a lower validation cross-entropy; the best epoch is kept.

**Logged each epoch:** train CE, JEPA, style MSE and contrast; validation CE, JEPA cosine, style MSE and contrast accuracy (how often the Titan latent is the closest of the three).

**Checkpoints, all mirrored to the Hub:**

| File | Holds | Saved |
|---|---|---|
| `ladder_<arm>_ckpt.pt` | the trained parts' weights, optimizer, scheduler, log, early-stopping state | after every epoch |
| `ladder_<arm>_best.pt` | the best epoch's trained weights | whenever validation CE improves |
| `ladder_<arm>_partial.json` | finished scoring parts | after each split, the slot test, the demo, the samples |
| `ladder_<arm>.json`, `decoder-lora-<arm>/`, `predictor-<arm>.pt` | the finished arm | at the end; the arm's training checkpoints are then removed from the Hub |

**After training:**
- scores on validation and test: chrF, facts, style rules, attribute error and Titan rate;
- the slot test on the first 50 test sentences;
- the chosen sentences, and 8 samples of one sentence.

**The predictor fix (after the first pass of the jepa arm).**
- In the first pass, the predictor wrote raw vectors and the cosine was taken after standardising them. In dimensions where the targets barely vary, that blew small offsets up, and at learning rate 1e-4 the predictor could not reach the target's scale. After one epoch the JEPA cosine was still −0.005, the same as control's.
- Now the predictor writes directly in the **standardised** latent space. The cosine compares that output with the standardised s(y), and Gemma reads it mapped back (ẑ × std + mean). The predictor's learning rate is 3e-4.
- **Control is kept from the first pass** (raw predictor output, learning rate 1e-4, files `ladder_control*`). Gemma ignored its ẑ (swapping slots changed 6% of outputs), so that difference doesn't affect the baseline. The jepa, style and contrast arms run after the fix (files `ladder2_*`).

**Predictor warm-up (jepa, style and contrast arms).**
- Before joint training, the predictor and style head train on the arm's latent losses **alone**, with Gemma untouched: JEPA, plus style at weight 0.3, plus contrast. They train on cached x prefixes, so this takes minutes.
- Training stops when the validation latent loss hasn't improved for 4 epochs (at most 40), and the best epoch is kept (`ladder2_<arm>_warm.pt`, mirrored).
- Why: in the first pass the JEPA cosine started rising only at epoch 3 (0.085, then 0.108), after the epoch that early stopping kept (epoch 2). With the warm-up, ẑ is already a prediction of the target when Gemma starts learning to read it.

**Running the arms in parallel.** `RUN_ARMS` (environment, default: all) chooses the arms a job runs. Control always loads from the Hub. On Modal, each of jepa, style and contrast runs as its own job on one GPU, and Cell 13 then compares control with that job's arm. The full ladder is put together from the results on the Hub.

In [ ]:
# Cell 8 — One arm: training, scores, slot test, chosen sentences, samples (checkpointed, mirrored to the Hub)

MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 6, 2, 4, 2
SAMPLE_SENTENCE, N_SAMPLES, SAMPLE_TEMPERATURE = "I'll be late to the office because of traffic.", 8, 0.9
STAGE1_STATE = snapshot()                   # stage-1 encoder/bridge and the initial predictor + style head: every arm starts here
SLOT_TEST_N = 50
SAMPLES = [VAL[0], VAL[min(4, len(VAL) - 1)], VAL[min(8, len(VAL) - 1)]]
TRAINED = [p for g in STAGE_GROUPS[2] for p in GROUPS[g]]


def train_state():
    return [p.detach().to("cpu", copy=True) for p in TRAINED]


def load_train_state(state):
    for p, s in zip(TRAINED, state):
        p.data.copy_(s.to(p.device))


def save(obj, path, json_file=False):
    if json_file:
        json.dump(obj, open(path, "w"), ensure_ascii=False)
    else:
        torch.save(obj, path)
    hub_put(path)


def slot_test(records):
    """Does Gemma use ẑ? Outputs with ẑ swapped for the next sentence's, and with noise at half ẑ's scale."""
    base = [translate(r)["output"] for r in records]
    swap = [translate(r, slot_record=records[(i + 1) % len(records)])["output"] for i, r in enumerate(records)]
    torch.manual_seed(SEED)
    noisy = [translate(r, noise=0.5)["output"] for r in records]
    ref = lambda i: records[i]["references"]
    return {"n": len(records),
            "changed_by_swap": round(sum(a != b for a, b in zip(base, swap)) / len(records), 3),
            "changed_by_noise": round(sum(a != b for a, b in zip(base, noisy)) / len(records), 3),
            "chrf_base": round(statistics.mean(max(chrf(o, x) for x in ref(i)) for i, o in enumerate(base)), 1),
            "chrf_swapped": round(statistics.mean(max(chrf(o, x) for x in ref(i)) for i, o in enumerate(swap)), 1),
            "chrf_noisy": round(statistics.mean(max(chrf(o, x) for x in ref(i)) for i, o in enumerate(noisy)), 1),
            "examples": [{"source": records[i]["source"], "base": base[i], "swapped": swap[i], "noisy": noisy[i]}
                         for i in range(min(4, len(records)))]}


XP = {}                                     # cached x prefixes (frozen encoder + bridge) for the warm-up


@torch.no_grad()
def cache_prefixes(records, batch=32):
    eval_mode()
    for i in range(0, len(records), batch):
        chunk = records[i:i + batch]
        for r, p in zip(chunk, prefixes([r["enc_input"] for r in chunk])):
            XP[r["id"]] = p.cpu()


def latent_losses(z, records, w):
    terms = {"jepa": (1 - jepa_cos(z, records)).mean(), "style": style_loss(z, records),
             "contrast": contrast_loss(z, records)}
    return sum(w[k] * v for k, v in terms.items() if v is not None and w[k]), terms


def warm_up(name, w, warm_file):
    """Train the predictor and style head on the arm's latent losses alone (Gemma untouched) until validation stops
    improving, so ẑ is already a prediction of the target when Gemma starts learning to read it."""
    if hub_get(warm_file):
        st = torch.load(warm_file, weights_only=False)
        predictor.load_state_dict(st["predictor"])
        style_head.load_state_dict(st["style_head"])
        print(f"[{name}] warm-up loaded: {st['log'][-1]}")
        return st["log"]
    if not XP:
        cache_prefixes(TRAIN + VAL)
    opt = torch.optim.AdamW(list(predictor.parameters()) + list(style_head.parameters()), lr=PRED_LR, weight_decay=0.0)
    rng = random.Random(SEED)

    def epoch_pass(rows, train):
        predictor.train(train)
        sums = Counter()
        for i in range(0, len(rows), WARM_BATCH):
            b = rows[i:i + WARM_BATCH]
            with torch.set_grad_enabled(train):
                z = predictor([XP[r["id"]].to(ENC_DEVICE) for r in b]) * Z_SPACE["std"] + Z_SPACE["mean"]
                loss, terms = latent_losses(z, b, w)
                if train:
                    opt.zero_grad()
                    loss.backward()
                    opt.step()
            sums["loss"] += loss.item() * len(b)
            sums["jepa_cos"] += (1 - terms["jepa"].item()) * len(b)
            sums["style_mse"] += terms["style"].item() * len(b)
            sims = contrast_sims(z.detach(), b)
            if sims is not None:
                sums["contrast_wins"] += int((sims.argmax(1) == 0).sum())
                sums["contrast_n"] += len(sims)
        out = {k: round(sums[k] / len(rows), 4) for k in ("loss", "jepa_cos", "style_mse")}
        out["contrast_acc"] = round(sums["contrast_wins"] / sums["contrast_n"], 3) if sums["contrast_n"] else None
        return out

    log, best, since, t0 = [], (float("inf"), 0, None), 0, time.perf_counter()
    for ep in range(1, WARM_MAX_EPOCHS + 1):
        torch.manual_seed(SEED + 1000 + ep)
        tr = epoch_pass(rng.sample(TRAIN, len(TRAIN)), True)
        va = epoch_pass(VAL, False)
        log.append({"epoch": ep, "train": tr, "validation": va})
        if va["loss"] < best[0]:
            best, since = (va["loss"], ep, {"predictor": {k: v.detach().clone() for k, v in predictor.state_dict().items()},
                                             "style_head": {k: v.detach().clone() for k, v in style_head.state_dict().items()}}), 0
        else:
            since += 1
        if ep == 1 or ep % 5 == 0 or since >= WARM_PATIENCE:
            print(f"[{name}] warm-up {ep}: train {tr} | validation {va} | {time.perf_counter() - t0:.0f}s")
        if since >= WARM_PATIENCE:
            break
    predictor.load_state_dict(best[2]["predictor"])
    style_head.load_state_dict(best[2]["style_head"])
    print(f"[{name}] warm-up kept epoch {best[1]}: {log[best[1] - 1]['validation']}")
    save({**best[2], "log": log}, warm_file)
    return log


def run_arm(name):
    w = ARMS[name]
    tag = "ladder" if name == "control" else "ladder2"      # control: first pass, kept; the rest: after the predictor fix
    out_file = f"{WORK}/{tag}_{name}.json"
    ck_file, best_file, part_file = (f"{WORK}/{tag}_{name}_ckpt.pt", f"{WORK}/{tag}_{name}_best.pt",
                                     f"{WORK}/{tag}_{name}_partial.json")
    if hub_get(out_file):
        print(f"[{name}] already done; results loaded from {out_file}")
        return json.load(open(out_file))
    for f in (ck_file, best_file, part_file):
        hub_get(f)
    restore(STAGE1_STATE)
    reset_decoder_lora()
    set_trainable(2)
    warm_log = warm_up(name, w, f"{WORK}/{tag}_{name}_warm.pt") if any(w.values()) else None
    steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
    opt, sched = make_optimizer(steps, 2)
    if os.path.exists(ck_file):
        ck = torch.load(ck_file, weights_only=False)
        load_train_state(ck["params"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        log, best, since_best = ck["log"], ck["best"], ck["since_best"]
        del ck
        gc.collect()
        print(f"[{name}] resuming after epoch {log[-1]['epoch']}")
    else:
        log = [{"epoch": 0, **val_losses(VAL)}]
        best, since_best = (log[0]["validation_ce"], 0), 0
        save(train_state(), best_file)
        print(f"[{name}] weights {w}", log[-1])
    t0 = time.perf_counter()
    for epoch in range(log[-1]["epoch"] + 1, MAX_EPOCHS + 1):
        if since_best >= PATIENCE:
            break
        torch.manual_seed(SEED + epoch)
        order = random.Random(SEED + epoch).sample(TRAIN, len(TRAIN))
        batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
        run = {"ce": [], "jepa": [], "style": [], "contrast": []}
        train_mode(2)
        for n, b in enumerate(batches, 1):
            ce, z = loss_on(b)
            terms = {"jepa": (1 - jepa_cos(z, b)).mean(), "style": style_loss(z, b), "contrast": contrast_loss(z, b)}
            loss = ce
            for k, v in terms.items():
                if v is not None:
                    run[k].append(v.item())
                    if w[k]:
                        loss = loss + w[k] * v.to(DEC_DEVICE)
            (loss / ACCUM).backward()
            run["ce"].append(ce.item())
            if n % ACCUM == 0 or n == len(batches):
                opt.step()
                sched.step()
                opt.zero_grad()
        log.append({"epoch": epoch, **{f"train_{k}": round(statistics.mean(v), 4) for k, v in run.items() if v},
                    **val_losses(VAL)})
        print(f"[{name}]", log[-1], f"| {time.perf_counter() - t0:.0f}s")
        if w["jepa"] and log[-1]["validation_jepa_cos"] < 0.05:
            print(f"[{name}] WARNING: validation JEPA cosine {log[-1]['validation_jepa_cos']} is still near zero: "
                  "the predictor is not learning the target")
        for r in SAMPLES:
            print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
        if log[-1]["validation_ce"] < best[0]:
            best, since_best = (log[-1]["validation_ce"], epoch), 0
            save(train_state(), best_file)
        else:
            since_best += 1
        save({"params": train_state(), "opt": opt.state_dict(), "sched": sched.state_dict(), "log": log,
              "best": best, "since_best": since_best}, ck_file)
        if since_best >= PATIENCE:
            print(f"[{name}] early stop: no lower validation cross-entropy for {PATIENCE} epochs")
    load_train_state(torch.load(best_file, weights_only=False))
    kept = best[1]
    print(f"[{name}] kept epoch {kept} (validation cross-entropy {best[0]})")
    del opt, sched
    gc.collect()
    torch.cuda.empty_cache()

    part = json.load(open(part_file)) if os.path.exists(part_file) else {"results": {}, "scores": {}}
    for split in EVAL_SPLITS:
        if split not in part["scores"]:
            part["results"][split], part["scores"][split] = evaluate(SPLITS[split])
            save(part, part_file, json_file=True)
        print(f"[{name}]", split, part["scores"][split])
    if "slot_test" not in part:
        part["slot_test"] = slot_test(SPLITS[VERDICT_SPLIT][:SLOT_TEST_N])
        save(part, part_file, json_file=True)
    print(f"[{name}] slot test:", {k: v for k, v in part["slot_test"].items() if k != "examples"})
    if "demo" not in part:
        part["demo"] = [{"input": s, **translate({"enc_input": f"{NAME[PERSONA]}: {s}"})} for s in DEMO_INPUTS]
        part["sampled"] = [o["output"] for o in generate({"enc_input": f"{NAME[PERSONA]}: {SAMPLE_SENTENCE}"},
                                                         n=N_SAMPLES, temperature=SAMPLE_TEMPERATURE)]
        save(part, part_file, json_file=True)
    dec_model.save_pretrained(f"{WORK}/decoder-lora-{name}")
    torch.save({"predictor": predictor.state_dict(), "style_head": style_head.state_dict()},
               f"{WORK}/predictor-{name}.pt")
    hub_put(f"{WORK}/decoder-lora-{name}")
    hub_put(f"{WORK}/predictor-{name}.pt")
    res = {"arm": name, "weights": w, "warm_log": warm_log, "log": log, "kept_epoch": kept, **part}
    save(res, out_file, json_file=True)
    for f in (ck_file, best_file):
        hub_delete(f)
    print(f"\n[{name}] examples:")
    show([x for s in EVAL_SPLITS for x in part["results"][s]], 6)
    print(f"[{name}] chosen sentences:")
    for k, d in enumerate(part["demo"], 1):
        print(f"{k:2d}. in : {d['input']}\n    out: {d['output']}" + ("" if d["stopped"] else "   [did not stop]"))
    print(f"\n[{name}] {N_SAMPLES} samples at temperature {SAMPLE_TEMPERATURE}: {SAMPLE_SENTENCE}")
    for o in part["sampled"]:
        print("   -", o)
    return res


RESULTS = {}

## Cell 9 — Arm: control (CE only)

In [ ]:
# Cell 9 — Arm: control

RESULTS["control"] = run_arm("control")

## Cell 10 — Arm: jepa (+ JEPA)

In [ ]:
# Cell 10 — Arm: jepa

if "jepa" in RUN_ARMS:
    RESULTS["jepa"] = run_arm("jepa")

## Cell 11 — Arm: style (+ style head, arm D)

In [ ]:
# Cell 11 — Arm: style

if "style" in RUN_ARMS:
    RESULTS["style"] = run_arm("style")

## Cell 12 — Arm: contrast (+ InfoNCE, arm E)

In [ ]:
# Cell 12 — Arm: contrast

if "contrast" in RUN_ARMS:
    RESULTS["contrast"] = run_arm("contrast")

## Cell 13 — The ladder side by side, and the verdict

Shown here:
- scores per split and arm, including style (attribute error, lower is better; Titan rate, higher is better), with the reference targets' style scores as the ceiling;
- the training curves and the slot test;
- each chosen sentence across the four arms, and each arm's samples;
- the **verdict per rung**: each arm against the arm below it, and against control. chrF on test, and facts pooled over all held-out rows.

In [ ]:
# Cell 13 — The ladder: control → jepa → style → contrast

ORDER = [a for a in ARMS if a in RESULTS]
cols = ["chrf"] + FAITH + STYLE + ["attr_err", "titan_rate", "copies_of_source", "distinct_outputs"]
print(f"{'split':11s} {'arm':9s} " + " ".join(f"{c[:12]:>12s}" for c in cols))
for split in EVAL_SPLITS:
    for arm in ORDER:
        s = RESULTS[arm]["scores"][split]
        print(f"{split:11s} {arm:9s} " + " ".join(f"{str(s.get(c)):>12s}" for c in cols))
print("reference targets:", REFERENCE, REF_STYLE)
print("\ntraining (per epoch):")
for arm in ORDER:
    print(f"  {arm} (kept epoch {RESULTS[arm]['kept_epoch']}):")
    for e in RESULTS[arm]["log"]:
        print("    ", {k: v for k, v in e.items()})
print("\nslot test (does Gemma use ẑ?):")
for arm in ORDER:
    print(f"  {arm:9s}", {k: v for k, v in RESULTS[arm]["slot_test"].items() if k != "examples"})
print("\nchosen sentences:")
for i, d in enumerate(RESULTS[ORDER[0]]["demo"]):
    print(f"in       : {d['input']}")
    for arm in ORDER:
        print(f"{arm:9s}: {RESULTS[arm]['demo'][i]['output']}")
    print()
for arm in ORDER:
    print(f"{arm} samples ({SAMPLE_SENTENCE}):")
    for o in RESULTS[arm]["sampled"]:
        print("   -", o)

pooled = {arm: summarise([x for s in EVAL_SPLITS for x in RESULTS[arm]["results"][s]]) for arm in ORDER}
print("\npooled over all held-out rows:",
      {arm: {k: pooled[arm].get(k) for k in ["chrf"] + FAITH + ["attr_err", "titan_rate"]} for arm in ORDER})


def verdict(arm, base):
    gain = RESULTS[arm]["scores"][VERDICT_SPLIT]["chrf"] - RESULTS[base]["scores"][VERDICT_SPLIT]["chrf"]
    worse = {k: (pooled[base][k], pooled[arm][k]) for k in FAITH
             if pooled[base][k] is not None and pooled[arm][k] is not None and pooled[arm][k] < pooled[base][k] - FAITH_SLACK}
    d_attr = pooled[arm]["attr_err"] - pooled[base]["attr_err"]
    d_titan = pooled[arm]["titan_rate"] - pooled[base]["titan_rate"]
    met = gain >= BAR_CHRF and not worse
    print(f"  {arm:9s} vs {base:8s}: {VERDICT_SPLIT} chrF {gain:+.1f} (bar +{BAR_CHRF}) | facts worse: {worse or 'none'} | "
          f"attribute error {d_attr:+.3f} | Titan rate {d_titan:+.3f} → {'BAR MET' if met else 'bar not met'}")


print("\nVERDICT")
for lower, upper in zip(ORDER, ORDER[1:]):
    verdict(upper, lower)
for arm in ORDER[2:]:
    verdict(arm, "control")

## Cell 14 — Settings and records

Each arm's Gemma LoRA and predictor (with style head) are saved (`decoder-lora-<arm>/`, `predictor-<arm>.pt`) and mirrored to `skamalj/titan-stage1` under `runs/<RUN_NAME>/`. The encoder LoRA and bridge are R16's (`r16/`).

In [ ]:
# Cell 14 — Settings and records

print(json.dumps({"revisions": REV, "data": DATA, "style_data": STYLE_DATA, "splits": {s: len(r) for s, r in SPLITS.items()},
                  "arms": ARMS, "slots": [K_SLOTS, K_STYLE], "contrast_tau": CONTRAST_TAU,
                  "predictor": {"layers": PRED_LAYERS, "width": PRED_WIDTH, "heads": PRED_HEADS, "lr": PRED_LR},
                  "lrs": LRS, "stage1": S1_SOURCE, "stage1_log": S1_LOG, "stage1_check": S1_CHECK,
                  "probes": {"attr_alpha": a_alpha, "attr_val_mae": -a_mae, "class_alpha": c_alpha, "class_val_acc": c_acc},
                  "reference_checks": REFERENCE, "reference_style": REF_STYLE,
                  "bar": {"split": VERDICT_SPLIT, "chrf": BAR_CHRF, "faith_slack": FAITH_SLACK}, "run": RUN_NAME,
                  "arms_summary": {a: {"log": RESULTS[a]["log"], "kept_epoch": RESULTS[a]["kept_epoch"],
                                       "scores": RESULTS[a]["scores"],
                                       "slot_test": RESULTS[a]["slot_test"]} for a in RESULTS}}, indent=1,
                 ensure_ascii=False))
print("\n----- RECORDS (JSON lines) -----")
for a in RESULTS:
    for split, rows in RESULTS[a]["results"].items():
        for x in rows:
            print(json.dumps({"arm": a, "split": split, **x}, ensure_ascii=False))
print("----- END RECORDS -----")

## Cell 15 — Oracle-slot diagnostic (only when `ORACLE_ARM` is set)

Does a perfect latent help? This cell reloads one finished arm's Gemma LoRA, predictor and style head from the Hub. It then generates the first 50 test sentences twice:
- with the **predicted** content slots ẑ (these must reproduce the arm's saved test outputs, which checks the reload);
- with the **true** latent s(y) of the reference in the 8 content slots (an oracle; the style slots stay predicted).

If the oracle lines are clearly better, the latent carries what the output needs, and the bottleneck is prediction quality (validation cosine 0.23). If they are not, this latent is not worth predicting. Results go to `oracle_<arm>.json`, mirrored to the Hub.

In [ ]:
# Cell 15 — Oracle-slot diagnostic

ORACLE_ARM = os.environ.get("ORACLE_ARM")
if ORACLE_ARM:
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict

    restore(STAGE1_STATE)
    set_trainable(2)
    lora_dir, pred_file = f"{WORK}/decoder-lora-{ORACLE_ARM}", f"{WORK}/predictor-{ORACLE_ARM}.pt"
    assert hub_get(lora_dir) and hub_get(pred_file), "the arm's saved weights are not on the Hub"
    res = set_peft_model_state_dict(dec_model, load_file(f"{lora_dir}/adapter_model.safetensors", device=str(DEC_DEVICE)))
    assert not [k for k in res.unexpected_keys if "lora_" in k], res.unexpected_keys
    st = torch.load(pred_file, map_location=ENC_DEVICE, weights_only=False)
    predictor.load_state_dict(st["predictor"])
    style_head.load_state_dict(st["style_head"])

    @torch.no_grad()
    def generate_slots(r, content=None):
        """Greedy translation with the predicted slots, or with `content` (K_SLOTS × D) in the content slots."""
        eval_mode()
        x = prefixes([r["enc_input"]])
        z = predictor(x) * Z_SPACE["std"] + Z_SPACE["mean"]
        if content is not None:
            z = z.clone()
            z[0, :K_SLOTS] = content.to(z.device, z.dtype)
        p = torch.cat([x[0], z[0]]).to(DEC_DEVICE)
        ids = torch.tensor([[START] + [PLACEHOLDER] * len(p)], device=DEC_DEVICE)
        SLOT.vectors = p
        out = dec_model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), do_sample=False, num_beams=1,
                                 max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS, pad_token_id=PAD,
                                 temperature=None, top_p=None, top_k=None)
        assert SLOT.vectors is None, "the prefix was not written into the prompt"
        return dec_tok.decode(out[0, ids.shape[1]:].tolist(), skip_special_tokens=True).strip()

    saved = {x["id"]: x["output"] for x in json.load(open(f"{WORK}/ladder2_{ORACLE_ARM}.json"))["results"]["test"]} \
        if hub_get(f"{WORK}/ladder2_{ORACLE_ARM}.json") else {}
    rows = []
    for r in SPLITS[VERDICT_SPLIT][:SLOT_TEST_N]:
        pred, orac = generate_slots(r), generate_slots(r, TARGETS[r["target"]])
        rows.append({"id": r["id"], "source": r["source"], "reference": r["target"], "predicted": pred, "oracle": orac,
                     "chrf_predicted": round(chrf(pred, r["target"]), 1), "chrf_oracle": round(chrf(orac, r["target"]), 1),
                     **{f"pred_{k}": v for k, v in faithfulness(r["source"], pred).items()},
                     **{f"orac_{k}": v for k, v in faithfulness(r["source"], orac).items()}})
    reproduced = sum(saved.get(x["id"]) == x["predicted"] for x in rows) / len(rows)
    summary = {"arm": ORACLE_ARM, "n": len(rows), "reproduced_saved_outputs": round(reproduced, 3),
               "chrf_predicted": round(statistics.mean(x["chrf_predicted"] for x in rows), 1),
               "chrf_oracle": round(statistics.mean(x["chrf_oracle"] for x in rows), 1),
               "changed_by_oracle": round(sum(x["predicted"] != x["oracle"] for x in rows) / len(rows), 3)}
    for k in FAITH:
        for side in ("pred", "orac"):
            v = [x[f"{side}_{k}"] for x in rows if x[f"{side}_{k}"] is not None]
            summary[f"{side}_{k}"] = round(sum(v) / len(v), 3) if v else None
    print("ORACLE SUMMARY", json.dumps(summary))
    save({"summary": summary, "rows": rows}, f"{WORK}/oracle_{ORACLE_ARM}.json", json_file=True)
    for x in rows[:8]:
        print(f"\nsrc : {x['source']}\npred: {x['predicted']}\norac: {x['oracle']}\nref : {x['reference']}")
